In [3]:
import os
import pandas as pd

basin_path = '/Users/milliespencer/Desktop/CR2_OGGM_Paper/cuencasoctubre(1999)'

basin_folders = sorted([
    f for f in os.listdir(basin_path)
    if os.path.isdir(f'{basin_path}/{f}')])

# ── Extract catchment areas from Q_med.csv ────────────────────────────────
catchment_areas = {}
basin_names     = {}

for basin_id in basin_folders:
    q_file = f'{basin_path}/{basin_id}/Q_med.csv'
    if os.path.exists(q_file):
        q = pd.read_csv(q_file)
        catchment_areas[int(basin_id)] = q['area_basin'].iloc[0]
        basin_names[int(basin_id)]     = q['name'].iloc[0]

print(f'Found catchment areas for {len(catchment_areas)} basins')

# ── Check COD_CUEN in RGI_BNA ─────────────────────────────────────────────
rgi_df = pd.read_csv(
    '/Users/milliespencer/Desktop/CR2_OGGM_Paper/'
    'files_chile_OGGM_climate_comparison/RGI_BNA_Clusters.csv')

print('\nCOD_CUEN raw sample:', rgi_df['COD_CUEN'].dropna().unique()[:10])
print('dtype:', rgi_df['COD_CUEN'].dtype)

# Check overlap after cleaning
cod_vals = rgi_df['COD_CUEN'].dropna().astype(str).str.strip()
print('\nCleaned COD_CUEN sample:', cod_vals.unique()[:10])

folder_ids = set(basin_folders)
cod_ids    = set(cod_vals.unique())
print(f'\nCOD_CUEN unique values: {len(cod_ids)}')
print(f'Basin folders: {len(folder_ids)}')
print(f'Overlap: {len(cod_ids & folder_ids)}')
print(f'In folders but not COD_CUEN: {folder_ids - cod_ids}')
print(f'In COD_CUEN but not folders: {(cod_ids - folder_ids)}')

Found catchment areas for 27 basins

COD_CUEN raw sample: ['129' '128' ' ' '127' '125' '124' '122' '120' '123' '126']
dtype: object

Cleaned COD_CUEN sample: ['129' '128' '' '127' '125' '124' '122' '120' '123' '126']

COD_CUEN unique values: 46
Basin folders: 44
Overlap: 26
In folders but not COD_CUEN: {'34', '10', '94', '54', '57', '91', '60', '45', '83', '71', '73', '81', '26', '43', '23', '38', '30', '21'}
In COD_CUEN but not folders: {'', '030', '026', '034', '021', '081', '091', '073', '045', '060', '038', '083', '071', '057', '094', '010', '043', '023', '054', '024'}


In [4]:
# Fix: strip leading zeros from COD_CUEN to match basin folder names
rgi_df['COD_CUEN_clean'] = rgi_df['COD_CUEN'].astype(str).str.strip().str.lstrip('0')
rgi_df['COD_CUEN_clean'] = rgi_df['COD_CUEN_clean'].replace('', None)

cod_ids    = set(rgi_df['COD_CUEN_clean'].dropna().unique())
folder_ids = set(basin_folders)
print(f'After fix — Overlap: {len(cod_ids & folder_ids)}')
print(f'Still missing: {folder_ids - cod_ids}')

# ── Map basins to clusters ────────────────────────────────────────────────
clusters = ['OT3','DA1','DA2','DA3','WA1','WA2','WA3','WA4','WA5','WA6']

basin_to_cluster = {}
for basin_id in folder_ids:
    basin_glaciers = rgi_df[
        rgi_df['COD_CUEN_clean'] == basin_id]['RGIId'].tolist()
    for cluster in clusters:
        cluster_glaciers = set(
            rgi_df[rgi_df['Cluster'] == cluster]['RGIId'].tolist())
        overlap = set(basin_glaciers) & cluster_glaciers
        if overlap:
            basin_to_cluster[basin_id] = cluster
            break

print('\nBasin → Cluster mapping:')
for bid in sorted(basin_to_cluster.keys(), key=lambda x: int(x)):
    area = catchment_areas.get(int(bid), 'no Q_med')
    name = basin_names.get(int(bid), '?')
    print(f'  Basin {bid:>4} ({name:<30}) → {basin_to_cluster[bid]}, '
          f'catchment={area} km²')

# ── Aggregate catchment area per cluster ─────────────────────────────────
cluster_catchment_km2 = {c: 0.0 for c in clusters}
for basin_id, cluster in basin_to_cluster.items():
    if int(basin_id) in catchment_areas:
        cluster_catchment_km2[cluster] += catchment_areas[int(basin_id)]

print('\nTotal catchment area per cluster (km²):')
for cluster in clusters:
    print(f'  {cluster}: {cluster_catchment_km2[cluster]:.1f} km²')

After fix — Overlap: 44
Still missing: set()

Basin → Cluster mapping:
  Basin   10 (Altiplanicas                  ) → OT3, catchment=467.556 km²
  Basin   21 (Rio Loa                       ) → DA1, catchment=23950.04 km²
  Basin   23 (?                             ) → DA1, catchment=no Q_med km²
  Basin   26 (?                             ) → DA1, catchment=no Q_med km²
  Basin   30 (te del Pacifico               ) → DA1, catchment=12310.98 km²
  Basin   34 (Rio Copiapo                   ) → DA1, catchment=18549.909 km²
  Basin   38 (?                             ) → DA1, catchment=no Q_med km²
  Basin   43 (Rio Elqui                     ) → DA1, catchment=9400.649 km²
  Basin   45 (Rio Limari                    ) → DA1, catchment=11422.59 km²
  Basin   54 (?                             ) → DA2, catchment=no Q_med km²
  Basin   57 (Rio Maipo                     ) → DA2, catchment=4839.047 km²
  Basin   60 (Rio Rapel                     ) → DA3, catchment=6264.944 km²
  Basin   71 (Rio

In [5]:
# Check what files exist in basins WITHOUT Q_med
missing_qmed = [b for b in basin_folders 
                if not os.path.exists(f'{basin_path}/{b}/Q_med.csv')]
print(f'Basins without Q_med ({len(missing_qmed)}): {missing_qmed}')

# Look at contents of a few missing ones
for basin_id in missing_qmed[:3]:
    bpath = f'{basin_path}/{basin_id}'
    print(f'\nBasin {basin_id} contents:')
    for f in os.listdir(bpath):
        size = os.path.getsize(f'{bpath}/{f}')
        print(f'  {f} ({size/1024:.1f} KB)')
        if f.endswith('.csv'):
            tmp = pd.read_csv(f'{bpath}/{f}', nrows=2)
            print(f'    Columns: {tmp.columns.tolist()}')

Basins without Q_med (17): ['101', '106', '107', '108', '116', '118', '119', '120', '121', '123', '126', '127', '129', '23', '26', '38', '54']

Basin 101 contents:
  fig_corr_mb_clima.png (133.8 KB)
  melt_rain_monthly.csv (14.2 KB)
    Columns: ['date', 'month', 'year', 'melt_on_glacier_sum', 'liq_prcp_on_glacier_sum']
  temp.csv (289.3 KB)
    Columns: ['RGI60-17.12439', 'RGI60-17.12442', 'RGI60-17.12443', 'RGI60-17.12444', 'RGI60-17.12445', 'RGI60-17.12446', 'RGI60-17.12447', 'RGI60-17.12448', 'RGI60-17.12449', 'RGI60-17.12450', 'RGI60-17.12451', 'RGI60-17.12452', 'RGI60-17.12453', 'RGI60-17.12454', 'RGI60-17.12455', 'RGI60-17.12456', 'RGI60-17.12457', 'RGI60-17.12458', 'RGI60-17.12459', 'RGI60-17.12460', 'RGI60-17.12461', 'RGI60-17.12462', 'RGI60-17.12463', 'RGI60-17.12507', 'RGI60-17.12508', 'RGI60-17.12510', 'RGI60-17.12511', 'RGI60-17.12512', 'RGI60-17.12513', 'RGI60-17.12514', 'RGI60-17.12515', 'RGI60-17.12517', 'RGI60-17.12526', 'RGI60-17.12527', 'RGI60-17.12535']
  melt_on_gl